In [3]:
from langgraph.graph import StateGraph,START,END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict
import time

In [5]:


# 1. Define the workflow state schema
class State(TypedDict):
    input: str
    step_1: str
    step_2: str
    step_3: str

# 2. Define the graph nodes (tasks)
def node_1(state: State):
    print("--- Executing Step 1 ---")
    return {"step_1": "Done"}

def node_2(state: State):
    print("--- Executing Step 2 (Simulating a long-running task/crash point) ---")
    # Deliberate delay of 30 seconds to allow manual interruption (crash simulation)
    time.sleep(30) 
    return {"step_2": "Done"}

def node_3(state: State):
    print("--- Executing Step 3 ---")
    return {"step_3": "Done"}

# 3. Build the StateGraph
workflow = StateGraph(State)

workflow.add_node("step_1_node", node_1)
workflow.add_node("step_2_node", node_2)
workflow.add_node("step_3_node", node_3)

# Define edges
workflow.set_entry_point("step_1_node")
workflow.add_edge("step_1_node", "step_2_node")
workflow.add_edge("step_2_node", "step_3_node")
workflow.add_edge("step_3_node", END)

# 4. Compile the graph with a MemorySaver checkpointer (enables persistence)
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

# 5. Define configuration with a Thread ID
config = {"configurable": {"thread_id": "fault_tolerance_thread_1"}}




In [6]:
# --- FIRST RUN: Trigger the workflow ---
# If you interrupt this execution during Step 2 (e.g., via keyboard interrupt / crash), 
# the progress up to Step 1 is safely saved in the checkpointer.
try:
    print("Starting workflow execution...")
    result = app.invoke({"input": "Start Workflow"}, config)
    print("Workflow completed successfully:", result)
except KeyboardInterrupt:
    print("\n[!] CRASH SIMULATED: Execution manually interrupted during Step 2!")



Starting workflow execution...
--- Executing Step 1 ---
--- Executing Step 2 (Simulating a long-running task/crash point) ---

[!] CRASH SIMULATED: Execution manually interrupted during Step 2!


In [7]:
# --- CHECK STATE AFTER CRASH ---
# You can check where the workflow stopped:
current_state = app.get_state(config)
print("\nCurrent State after crash:", current_state.values)
print("Next node to execute:", current_state.next)





Current State after crash: {'input': 'Start Workflow', 'step_1': 'Done'}
Next node to execute: ('step_2_node',)


In [8]:
# --- RESUME EXECUTION FROM THE CRASH POINT ---
# To resume, invoke the app with input=None and the SAME thread_id.
# LangGraph will automatically skip completed steps and resume directly at Step 2.
print("\nResuming workflow from the last checkpoint...")
resumed_result = app.invoke(None, config)
print("Resumed Workflow completed successfully:", resumed_result)


Resuming workflow from the last checkpoint...
--- Executing Step 2 (Simulating a long-running task/crash point) ---
--- Executing Step 3 ---
Resumed Workflow completed successfully: {'input': 'Start Workflow', 'step_1': 'Done', 'step_2': 'Done', 'step_3': 'Done'}
